<a href="https://colab.research.google.com/github/Kiranmai-Guddanti/BEE-102-Spring-2025-Assignment/blob/main/04_Viterbi_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Problem Statement**
---



**Writing Viterbi Algorithm for the Primer**

Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.

By doing the above two, you earn 1 mark.

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

# **Approach**


---




####  1. Model the Problem

We begin by defining three components of the HMM:

- **States**: These represent biological regions like exon (E), donor splice site (5), and intron (I).
- **Transition Probabilities**: These give the chance of moving from one state to another (e.g., from exon to splice site).
- **Emission Probabilities**: These define the likelihood of emitting a nucleotide (A, C, G, T) from each state.
- **Initial Probabilities**: These indicate where the sequence is most likely to begin.

These probabilities are either given or assumed based on biological knowledge (e.g., exon regions emit nucleotides equally).

\


####  2. Calculate Log Probability of a Known Path

To validate the model setup, we first write a function to compute the **log-probability of a known path emitting a given sequence**.

- For each position in the sequence:
  - Take the transition probability from the previous state to the current one.
  - Take the emission probability of the observed nucleotide in the current state.
  - Multiply these and take the logarithm.
- Add up all the log-values.
- If the last state is an intron, include the log-probability of transitioning to the end state.

This gives the overall log-probability of that known state path emitting the observed sequence.


\

####  3. Implement the Viterbi Algorithm

Next, we use the Viterbi algorithm to find the **most likely hidden path** without knowing it in advance.

The main idea is to use **dynamic programming**:

- We create a matrix where each row is a state and each column is a position in the DNA sequence.
- Each cell stores the **maximum log-probability** of reaching that state at that position.
- For each step in the sequence:
  - We look at all possible previous states.
  - For each, calculate the total log-probability to the current state.
  - Pick the path with the highest probability.
- We also store **backpointers** to trace the best path backward at the end.

After processing all positions, we identify the final state with the highest probability and backtrack using the pointers to reconstruct the best state sequence.

---

Using this structured approach, we are able to:

- Validate our model with known paths.
- Apply the Viterbi algorithm to **infer the hidden biological regions** from the observed nucleotide sequence.



In [26]:
import numpy as np
import math

# Define States and Nucleotides
states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

# Initial Probabilities: Starting probabilities for each state
initial_probs = {'E': 1.0, '5': 0.0, 'I': 0.0}

# Transition Probabilities between states
trans_probs = {
    'Start': {'E': 1.0, '5': 0.0, 'I': 0.0, 'End': 0.0},
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0, 'I': 0.9, 'End': 0.1}
}

# Emission Probabilities for each nucleotide given the state
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

In [27]:
# Log function: Handles 0 probabilities by returning -inf for log(0)
def log(x):
    return math.log(x) if x > 0 else -math.inf

# Function to calculate the log probability of a given path and observed sequence
def log_prob_given_path(state_path, sequence):
    log_prob = 0.0
    prev_state = 'Start'

    for i in range(len(sequence)):
        current_state = state_path[i]
        observed_nucleotide = sequence[i]
        log_prob += log(trans_probs[prev_state][current_state]) + log(emission_probs[current_state][observed_nucleotide])
        prev_state = current_state

    log_prob += log(trans_probs[prev_state]['End'])

    return round(log_prob,2)

  # Example State Path and Observed Sequence
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans = log_prob_given_path(state_path, observed_sequence)
print(f"Log probability of the given state path: {ans}")

Log probability of the given state path: -41.22


In [28]:
def viterbi_algorithm(sequence):
    T, N = len(sequence), len(states)
    V = np.full(N, -np.inf)
    B = np.zeros((T, N), dtype=int)

    for i, s in enumerate(states):
        V[i] = log(initial_probs[s]) + log(emission_probs[s][sequence[0]])

    for t in range(1, T):
        new_V = np.full(N, -np.inf)
        for j, sj in enumerate(states):
            probs = [V[i] + log(trans_probs[states[i]][sj]) for i in range(N)]
            B[t][j] = np.argmax(probs)
            new_V[j] = max(probs) + log(emission_probs[sj][sequence[t]])
        V = new_V

    idx = np.zeros(T, dtype=int)
    idx[-1] = np.argmax(V)
    for t in range(T - 1, 0, -1):
        idx[t - 1] = B[t][idx[t]]

    path = ''.join(states[i] for i in idx)
    return path, round(V[idx[-1]], 2)


In [29]:
observed_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
best_path, max_log_prob = viterbi_algorithm(observed_seq)
print(f"Most probable path: {best_path}\nlog probability: {max_log_prob}")

Most probable path: EEEEEEEEEEEEEEEEEEEEEEEEEE
log probability: -38.68
